# ERP003950 (мышь) — симуляция merged reads через InSilicoSeq

Строит `templates.fasta` + `read_counts.tsv` из уже смердженных (pRESTO
`AssemblePairs.py`, `*_assemble-pass.fastq.gz`) reads per sample, затем
прогоняет `iss generate --sequence_type amplicon` (InSilicoSeq 2.0), чтобы
получить синтетические R1/R2 reads с известной ground truth (каждый
сгенерированный read происходит от конкретной реальной merged-последовательности
с известным количеством копий).

Подход: dataset/template-based simulation (не de novo) — каждая уникальная
merged-последовательность сэмпла становится "геномом"-шаблоном для ISS,
`read_counts.tsv` задаёт её множественность = число раз, которое эта
последовательность встретилась в реальных merged reads. `--sequence_type
amplicon` фиксирует forward read на старте шаблона и reverse read на его
конце — это и есть ампликонная симуляция (не случайная фрагментация генома).

Работает на OneQ task `bbc68913-4463-433d-b647-c3b8a76555ba`, kernel **BCR
Pipeline** (`bcr_env`, где `insilicoseq` уже установлен).

Источник входных данных: `/data/user/epishkin/results/ERP003950/merged/fastq/`
(6 mouse samples ERR346596–ERR346601, уже полностью проаннотированы IgBLAST).


### 1. Env check

In [ ]:
import os, sys, sysconfig, subprocess

_ENV_CANDIDATES = [
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)

print(f"Using env: {_CONDA_ENV}")
iss_path = subprocess.run(["which", "iss"], capture_output=True, text=True).stdout.strip()
if not iss_path:
    raise RuntimeError("iss not found on PATH. Use BCR Pipeline kernel or `pip install insilicoseq` in bcr_env.")
ver = subprocess.run(["iss", "--version"], capture_output=True, text=True)
print("iss:", iss_path)
print(ver.stdout.strip() or ver.stderr.strip())


### 2. Config

In [ ]:
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]

MERGED_FASTQ_DIR = VOLUME / "results" / DATASET / "merged" / "fastq"

OUT_BASE = VOLUME / "results" / DATASET / "simulated" / "insilicoseq"
TEMPLATES_DIR = OUT_BASE / "templates"
FASTQ_DIR = OUT_BASE / "fastq"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
for d in (TEMPLATES_DIR, FASTQ_DIR, LOGS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- ручки ---
MIN_COPY_NUMBER = 1      # оставлять шаблоны, встретившиеся >= N раз в реальных merged reads (подними, чтобы уменьшить выход)
MODEL = "miseq"          # встроенная ISS-модель; для более точного соответствия ERP003950 используй свою .npz (через `iss model`)
SEQUENCE_TYPE = "amplicon"
NPROC = 8
SEED = 42
COMPRESS = True
FORCE = False


### 3. Build `templates.fasta` + `read_counts.tsv` per sample

Читает `{sample}_assemble-pass.fastq.gz`, схлопывает точные дубликаты
последовательностей (exact match). ID шаблона несёт количество копий в
самом имени только для читаемости лога, но авторитетный источник
множественности — `read_counts.tsv` (формат ISS: `template_id<TAB>read_count`,
без заголовка, обязателен при `--genomes`).


In [ ]:
import gzip, csv, time

def iter_fastq(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n")
            h.readline()  # +
            h.readline()  # качество
            yield seq

def build_templates_and_counts(sample, min_copy_number=MIN_COPY_NUMBER, force=FORCE):
    in_fastq = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
    templates_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    counts_tsv = TEMPLATES_DIR / f"{sample}_read_counts.tsv"

    if templates_fa.exists() and counts_tsv.exists() and not force:
        n_templates = sum(1 for _ in open(counts_tsv))
        print(f"[{sample}] [skip] templates already built: {n_templates:,} templates")
        return {"sample": sample, "status": "skipped"}

    if not in_fastq.exists():
        raise FileNotFoundError(f"Missing merged FASTQ for {sample}: {in_fastq}")

    t0 = time.time()
    counts = {}
    n_reads = 0
    for seq in iter_fastq(in_fastq):
        counts[seq] = counts.get(seq, 0) + 1
        n_reads += 1

    n_unique_all = len(counts)
    kept = [(seq, n) for seq, n in counts.items() if n >= min_copy_number]
    kept.sort(key=lambda x: -x[1])

    with open(templates_fa, "w") as fa, open(counts_tsv, "w", newline="") as tsv:
        writer = csv.writer(tsv, delimiter="\t")
        for idx, (seq, n) in enumerate(kept, start=1):
            tpl_id = f"{sample}_u{idx:07d}_n{n}"
            fa.write(f">{tpl_id}\n{seq}\n")
            writer.writerow([tpl_id, n])

    elapsed = time.time() - t0
    n_reads_kept = sum(n for _, n in kept)
    print(f"[{sample}] reads={n_reads:,} unique={n_unique_all:,} "
          f"kept_templates={len(kept):,} (>= {min_copy_number} copies, {n_reads_kept:,} reads) "
          f"elapsed={elapsed:.1f}s")
    return {
        "sample": sample, "status": "built",
        "input_reads": n_reads, "unique_sequences": n_unique_all,
        "templates_kept": len(kept), "reads_represented": n_reads_kept,
        "templates_fasta": str(templates_fa), "read_counts_tsv": str(counts_tsv),
    }


In [ ]:
template_rows = [build_templates_and_counts(sample) for sample in SAMPLES]

qc_path = QC_DIR / "templates_qc.tsv"
built_rows = [r for r in template_rows if r.get("status") == "built"]
if built_rows:
    with open(qc_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(built_rows[0].keys()), delimiter="\t")
        writer.writeheader()
        writer.writerows(built_rows)
    print(f"wrote {qc_path}")


### 4. Run `iss generate` per sample

`--sequence_type amplicon` + `--readcount_file` пропускает `--n_reads`
(количество ридов считается из суммы `read_counts.tsv`, т.е. равно числу
реальных merged reads, представленных сохранёнными шаблонами).
Длинные прогоны — heartbeat каждые 30с, PID/stdout/stderr в `logs/`.


In [ ]:
import subprocess, time

def run_iss_generate(sample, model=MODEL, sequence_type=SEQUENCE_TYPE, nproc=NPROC,
                      seed=SEED, compress=COMPRESS, force=FORCE):
    templates_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    counts_tsv = TEMPLATES_DIR / f"{sample}_read_counts.tsv"
    out_prefix = FASTQ_DIR / sample
    ext = ".fastq.gz" if compress else ".fastq"
    r1_out = Path(str(out_prefix) + f"_R1{ext}")
    r2_out = Path(str(out_prefix) + f"_R2{ext}")

    if r1_out.exists() and r2_out.exists() and not force:
        print(f"[{sample}] [skip] {r1_out.name} exists")
        return {"sample": sample, "status": "skipped"}

    if not templates_fa.exists() or not counts_tsv.exists():
        raise FileNotFoundError(f"Missing templates for {sample}; run step 3 first")

    stdout_path = LOGS_DIR / f"{sample}_iss.stdout.txt"
    stderr_path = LOGS_DIR / f"{sample}_iss.stderr.txt"

    cmd = [
        "iss", "generate",
        "--genomes", str(templates_fa),
        "--readcount_file", str(counts_tsv),
        "--sequence_type", sequence_type,
        "--model", model,
        "--cpus", str(nproc),
        "--output", str(out_prefix),
    ]
    if seed is not None:
        cmd += ["--seed", str(seed)]
    if compress:
        cmd.append("--compress")

    print(f"[{sample}] [run] {' '.join(cmd)}")
    t0 = time.time()
    with open(stdout_path, "w") as out_h, open(stderr_path, "w") as err_h:
        proc = subprocess.Popen(cmd, stdout=out_h, stderr=err_h, text=True)
        print(f"  pid={proc.pid} stdout={stdout_path.name} stderr={stderr_path.name}")
        while True:
            rc = proc.poll()
            if rc is not None:
                break
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(30)
    elapsed = time.time() - t0
    if rc != 0:
        raise RuntimeError(f"iss generate failed for {sample} (exit {rc}); see {stderr_path}")

    n_r1 = sum(1 for _ in iter_fastq(r1_out))
    print(f"[{sample}] done: R1={n_r1:,} reads elapsed={elapsed/60:.1f} min")
    return {
        "sample": sample, "status": "done", "elapsed_sec": f"{elapsed:.1f}",
        "reads_generated": n_r1, "r1_fastq": str(r1_out), "r2_fastq": str(r2_out),
    }


In [ ]:
sim_rows = [run_iss_generate(sample) for sample in SAMPLES]

qc_path = QC_DIR / "simulate_qc.tsv"
done_rows = [r for r in sim_rows if r.get("status") == "done"]
if done_rows:
    with open(qc_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(done_rows[0].keys()), delimiter="\t")
        writer.writeheader()
        writer.writerows(done_rows)
    print(f"wrote {qc_path}")

total_generated = sum(r.get("reads_generated", 0) for r in sim_rows if r.get("status") == "done")
print(f"total simulated read pairs: {total_generated:,}")


### Notes / upgrade path

- `--model miseq` — встроенная ISS KDE-модель (обучена не на ERP003950).
  Для более точного профиля ошибок под конкретно этот MiSeq 2x250 run
  можно натренировать `iss model` на реальных raw R1/R2 этого датасета
  (`bowtie2` alignment на merged reads как reference → `.npz`), см.
  `iss model --help`. Не сделано здесь намеренно (лишний шаг indexing/align).
- `MIN_COPY_NUMBER=1` сохраняет все уникальные merged-последовательности —
  это может дать очень много шаблонов (близко к числу input reads, т.к.
  merged reads до дедупликации почти все уникальны из-за sequencing error).
  Поднимите порог, если нужен компактный smoke-test запуск.
- Выход: `results/ERP003950/simulated/insilicoseq/fastq/{sample}_R1.fastq.gz`
  + `_R2.fastq.gz`, с ground truth в `templates/{sample}_read_counts.tsv`.
